# G18 — 3D Voronoi Orthopedic Cast Pipeline
## Step-by-Step Visual Walkthrough

> **AUM · College of Engineering & Technology · GP2 Spring 2026**  
> Supervised by Dr. Georges Zakka El Nashef

Run cells **top to bottom**. Supports three capture modes:

| `CAPTURE_MODE` | Meaning |
|---|---|
| `'synthetic'` | No hardware — generates arm photos programmatically (default) |
| `'real_rgb'` | Real photos from any camera — no depth data |
| `'real_rgbd'` | **Recommended** — Intel RealSense D435 RGB + depth |

```
/content/g18_project/
  inputs/                  arm photos (synthetic or real)
  real_capture/            unzipped from Google Drive (real modes)
    intrinsics.json
    background.png
    rgb/frame_000.png ...
    depth/frame_000.npy ...
  stage_outputs/
    stage_01_images/ ... stage_10_final/
  g18_unified_pipeline.py  upload before running
  G18_Visual_Walkthrough.html
```


In [ ]:
import subprocess, sys
pkgs = ['open3d','trimesh','manifold3d','scikit-image',
        'scipy','opencv-python-headless','matplotlib','numpy']
for p in pkgs:
    print(f'  {p}...', end=' ', flush=True)
    subprocess.run([sys.executable,'-m','pip','install',p,'-q'], check=False)
    print('ok')
print('All packages ready.')

## Cell 1 — Project Setup

In [ ]:
import os, sys, shutil
BASE      = '/content/g18_project'
INPUTS    = BASE + '/inputs'
STAGE_OUT = BASE + '/stage_outputs'
STAGE_DIRS = {k: STAGE_OUT+'/'+v for k,v in [
    ('S01','stage_01_images'),('S02','stage_02_silhouettes'),
    ('S03','stage_03_voxels'),('S04','stage_04_mesh'),
    ('S05','stage_05_hull'),  ('S06','stage_06_cleaned'),
    ('S07','stage_07_shell'), ('S08','stage_08_voronoi'),
    ('S09','stage_09_split'), ('S10','stage_10_final'),
]}
for d in [BASE,INPUTS,STAGE_OUT]+list(STAGE_DIRS.values()):
    os.makedirs(d, exist_ok=True)
pipeline_src = '/content/g18_unified_pipeline.py'
pipeline_dst = BASE + '/g18_unified_pipeline.py'
if os.path.exists(pipeline_src):
    shutil.copy(pipeline_src, pipeline_dst); print('Pipeline copied.')
elif os.path.exists(pipeline_dst):
    print('Pipeline already present.')
else:
    print('UPLOAD g18_unified_pipeline.py to /content/ then re-run this cell.')
if BASE not in sys.path: sys.path.insert(0, BASE)
print('Inputs :', INPUTS)
print('Outputs:', STAGE_OUT)

## Cell 2 — Import Pipeline

In [ ]:
import importlib.util, types, logging, copy, warnings, glob as _g
import numpy as np
import cv2
import trimesh
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from scipy import ndimage
from skimage import measure
warnings.filterwarnings('ignore')

for mod in ['pyrealsense2','open3d']:
    if mod not in sys.modules:
        sys.modules[mod] = types.ModuleType(mod)

spec = importlib.util.spec_from_file_location('pipe', pipeline_dst)
pipe = importlib.util.module_from_spec(spec)
logging.disable(logging.CRITICAL)
spec.loader.exec_module(pipe)
logging.disable(logging.NOTSET)

pipe.CONFIG.update({
    'image_dir': INPUTS,
    'background_image': INPUTS + '/_background.png',
    'output_dir': BASE + '/g18_output',
    'frames_dir': BASE + '/g18_output/frames',
    'mesh_dir':   BASE + '/g18_output/mesh',
    'log_file':   BASE + '/g18_output/pipeline.log',
    'silhouette_method': 'full',
    'voronoi_enabled': True, 'fastener_enabled': True, 'roi_enabled': False,
})
os.makedirs(pipe.CONFIG['output_dir'], exist_ok=True)
os.makedirs(pipe.CONFIG['mesh_dir'],   exist_ok=True)

plt.rcParams.update({'figure.facecolor':'#0F172A','axes.facecolor':'#1E293B',
    'axes.edgecolor':'#334155','text.color':'white','axes.labelcolor':'white',
    'xtick.color':'#94A3B8','ytick.color':'#94A3B8'})

def save_fig(path):
    plt.savefig(path, dpi=120, bbox_inches='tight', facecolor='#0F172A')
    plt.close()
    from IPython.display import Image, display
    print('  saved:', path)
    display(Image(path))

def plot_mesh_3d(ax, V, F, title, elev=20, azim=45, color='#3B82F6', alpha=0.4):
    sub = F[::max(1, len(F)//2000)]
    p = Poly3DCollection([V[f] for f in sub], alpha=alpha, linewidth=0.05)
    p.set_facecolor(color); p.set_edgecolor('#1E40AF')
    ax.add_collection3d(p)
    ax.set_xlim(V[:,0].min(), V[:,0].max())
    ax.set_ylim(V[:,1].min(), V[:,1].max())
    ax.set_zlim(V[:,2].min(), V[:,2].max())
    ax.view_init(elev=elev, azim=azim); ax.set_facecolor('#0F172A')
    ax.set_title(title, color='white', fontsize=10, pad=6)
    ax.tick_params(colors='#475569', labelsize=7)

print('Pipeline loaded and configured.')

---
## ═══ CAPTURE MODE SELECTION ═══
**Change `CAPTURE_MODE` here — this single variable controls the entire notebook.**

| Value | Description |
|---|---|
| `'synthetic'` | No hardware needed. Generates arm images programmatically. |
| `'real_rgb'` | Load real RGB photos (no depth). Use any camera. |
| `'real_rgbd'` | **Recommended.** RealSense D435 — RGB + depth silhouette. |

For `real_rgb` or `real_rgbd`: run `g18_capture.py` on your PC first,  
zip the output folder, upload to `My Drive/G18_Project/g18_capture.zip`.


In [ ]:
# ════════════════════════════════════════════════════════════════
# SET CAPTURE MODE HERE — change this one variable
# ════════════════════════════════════════════════════════════════
CAPTURE_MODE = 'real_rgbd'   # 'synthetic' | 'real_rgb' | 'real_rgbd'

# Google Drive paths (only used in real modes)
DRIVE_ZIP   = '/content/drive/MyDrive/G18_Project/g18_capture.zip'
CAPTURE_DIR = '/content/g18_project/real_capture'
BG_FILENAME = 'background.png'

print(f'CAPTURE_MODE = {CAPTURE_MODE!r}')
if CAPTURE_MODE == 'synthetic':
    print('  → Synthetic mode: no hardware or Drive upload needed.')
    print('  → Skip the Drive Mount cell below — go straight to patches.')
elif CAPTURE_MODE in ('real_rgb', 'real_rgbd'):
    print(f'  → Real capture mode: will load from {CAPTURE_DIR}')
    print(f'  → Make sure g18_capture.zip is in Google Drive first.')
    print(f'  → Run the Drive Mount cell below, then skip the patch cell.')
else:
    raise ValueError(f'Unknown CAPTURE_MODE: {CAPTURE_MODE!r}. Use synthetic/real_rgb/real_rgbd.')

---
## Drive Mount + Unzip
**Run this cell ONLY for `real_rgb` or `real_rgbd` modes.**  
Skip it if `CAPTURE_MODE = 'synthetic'`.

This cell:
1. Mounts Google Drive
2. Unzips `g18_capture.zip` → `/content/g18_project/real_capture/`
3. Reads `intrinsics.json` and updates pipeline CONFIG
4. Estimates camera-to-arm distance from depth data


In [ ]:
if CAPTURE_MODE != 'synthetic':
    from google.colab import drive
    import zipfile, json as _json

    # ── Mount Drive ──────────────────────────────────────────────────────
    drive.mount('/content/drive')
    assert os.path.exists(DRIVE_ZIP), (
        f'ZIP not found: {DRIVE_ZIP}\n'
        'Upload g18_capture.zip to My Drive/G18_Project/ first.')

    # ── Unzip ────────────────────────────────────────────────────────────
    if os.path.exists(CAPTURE_DIR):
        shutil.rmtree(CAPTURE_DIR)
    os.makedirs(CAPTURE_DIR, exist_ok=True)

    with zipfile.ZipFile(DRIVE_ZIP, 'r') as z:
        z.extractall(CAPTURE_DIR)

    # Handle case where zip wraps a sub-folder (user zipped the folder itself)
    inner = os.path.join(CAPTURE_DIR, 'g18_capture')
    if os.path.isdir(inner):
        for item in os.listdir(inner):
            shutil.move(os.path.join(inner, item), CAPTURE_DIR)
        os.rmdir(inner)

    # ── Verify files ─────────────────────────────────────────────────────
    rgb_files   = sorted(_g.glob(os.path.join(CAPTURE_DIR, 'rgb',   'frame_*.png')))
    depth_files = sorted(_g.glob(os.path.join(CAPTURE_DIR, 'depth', 'frame_*.npy')))
    bg_exists   = os.path.exists(os.path.join(CAPTURE_DIR, BG_FILENAME))
    intr_path   = os.path.join(CAPTURE_DIR, 'intrinsics.json')
    intr_exists = os.path.exists(intr_path)

    print(f'  RGB frames   : {len(rgb_files)}')
    print(f'  Depth frames : {len(depth_files)}')
    print(f'  background   : {bg_exists}')
    print(f'  intrinsics   : {intr_exists}')

    assert len(rgb_files) > 0, 'No rgb/frame_*.png files found after unzip.'
    if CAPTURE_MODE == 'real_rgbd':
        assert len(depth_files) == len(rgb_files), (
            f'RGB ({len(rgb_files)}) and depth ({len(depth_files)}) counts must match.')

    # ── Read intrinsics and update CONFIG ─────────────────────────────────
    if intr_exists:
        with open(intr_path) as f:
            real_intr = _json.load(f)
        pipe.CONFIG.update({
            'camera_fx' : real_intr['fx'],
            'camera_fy' : real_intr['fy'],
            'camera_cx' : real_intr['cx'],
            'camera_cy' : real_intr['cy'],
            'img_width' : real_intr['width'],
            'img_height': real_intr['height'],
        })
        print(f'  intrinsics loaded: fx={real_intr["fx"]:.1f}  '
              f'fy={real_intr["fy"]:.1f}  '
              f'cx={real_intr["cx"]:.1f}  cy={real_intr["cy"]:.1f}')

    # ── Estimate camera-arm distance from depth median ────────────────────
    if CAPTURE_MODE == 'real_rgbd' and len(depth_files) > 0:
        medians = []
        for dp in depth_files[:min(10, len(depth_files))]:  # sample 10 frames
            d = np.load(dp)
            arm = d[(d > 0.1) & (d < 1.0)]
            if len(arm) > 200:
                medians.append(float(np.median(arm)))
        if medians:
            cam_dist_m = float(np.median(medians))
            pipe.CONFIG['camera_distance_mm'] = cam_dist_m * 1000
            print(f'  Camera-arm distance estimated: {cam_dist_m*1000:.0f} mm')

    # ── Update num_images and voxel grid ──────────────────────────────────
    pipe.CONFIG.update({
        'num_images'     : len(rgb_files),
        'voxel_grid_size': 300,
    })
    print(f'  num_images={len(rgb_files)}  voxel_grid_size=300')
    print(f'  Config updated.')

else:
    print('Synthetic mode — Drive mount skipped.')

---

## SYNTHETIC PATCHES — Realistic Forearm + Relaxed Fist
**Run this cell ONLY for `CAPTURE_MODE = 'synthetic'`.**  
Skip it (or run — it is guarded) for real capture modes.

Monkey-patches the pipeline mesh builder, renderer and config — no file changes needed.  
- Yellow foreground on white background
- Forearm only (no elbow/upper arm)
- Relaxed closed fist: 4 fingers × 3 phalanges, MCP~25° PIP~75° DIP~45°
- Thumb angled 30° outward, curled toward palm
- 72 images, 300³ voxel grid


In [ ]:
import math

if CAPTURE_MODE == 'synthetic':
    # ─── GEOMETRY HELPERS ────────────────────────────────────────────────────

    def make_frustum(r_bot, r_top, length, sections=28):
        a  = np.linspace(0, 2*np.pi, sections, endpoint=False)
        vb = np.column_stack([r_bot*np.cos(a), np.zeros(sections),        r_bot*np.sin(a)])
        vt = np.column_stack([r_top*np.cos(a), np.full(sections, length), r_top*np.sin(a)])
        cb = np.array([[0., 0.,      0.]])
        ct = np.array([[0., length,  0.]])
        V  = np.vstack([vb, vt, cb, ct])
        N  = sections
        F  = []
        for i in range(N):
            j = (i+1) % N
            F += [[i,j,N+j],[i,N+j,N+i],[2*N,j,i],[2*N+1,N+i,N+j]]
        return trimesh.Trimesh(vertices=V, faces=np.array(F), process=True)

    def align_y_to_dir(direction):
        d = np.array(direction, dtype=float)
        d /= np.linalg.norm(d)
        y = np.array([0., 1., 0.])
        v = np.cross(y, d)
        s = np.linalg.norm(v)
        c = np.dot(y, d)
        R4 = np.eye(4)
        if s < 1e-6:
            if c < 0: R4[1,1] = -1; R4[2,2] = -1
        else:
            vx = np.array([[0,-v[2],v[1]],[v[2],0,-v[0]],[-v[1],v[0],0]])
            R4[:3,:3] = np.eye(3) + vx + vx@vx*((1-c)/(s*s))
        return R4

    def make_curled_finger(mcp_pos, x_spread, lengths, radii, curl_angles, sections=12):
        parts = []
        pos   = np.array(mcp_pos, dtype=float)
        total_curl = 0.0
        for k, (ln, r, ca) in enumerate(zip(lengths, radii, curl_angles)):
            total_curl += ca
            dy = -math.cos(total_curl)
            dz = -math.sin(total_curl)
            dx = math.sin(x_spread)
            d  = np.array([dx, dy, dz])
            d /= np.linalg.norm(d)
            jr  = r * (1.12 if k == 0 else 1.06)
            jnt = trimesh.creation.icosphere(subdivisions=2, radius=jr)
            jnt.apply_translation(pos)
            parts.append(jnt)
            r_tip = r * 0.86
            seg   = make_frustum(r_tip, r, ln, sections=sections)
            seg.apply_transform(align_y_to_dir(d))
            seg.apply_translation(pos)
            parts.append(seg)
            pos = pos + d * ln
        tip = trimesh.creation.icosphere(subdivisions=2, radius=radii[-1]*0.88)
        tip.apply_translation(pos)
        parts.append(tip)
        return parts

    def _build_realistic_arm_mesh():
        parts = []
        forearm = make_frustum(0.021, 0.034, 0.180, sections=36)
        forearm.apply_translation([0.000, -0.048, 0.000])
        parts.append(forearm)
        wrist = trimesh.creation.icosphere(subdivisions=2, radius=1.0)
        wrist.vertices[:,0] *= 0.028; wrist.vertices[:,1] *= 0.015; wrist.vertices[:,2] *= 0.012
        wrist.apply_translation([0.001, -0.053, 0.000])
        parts.append(wrist)
        palm = trimesh.creation.icosphere(subdivisions=3, radius=1.0)
        palm.vertices[:,0] *= 0.042; palm.vertices[:,1] *= 0.022; palm.vertices[:,2] *= 0.014
        palm.apply_translation([0.001, -0.080, 0.001])
        parts.append(palm)
        y_mcp = -0.100
        finger_defs = [
            (-0.022, -0.10, [0.044,0.027,0.020], [0.0080,0.0072,0.0062],
             [math.radians(25), math.radians(75), math.radians(45)]),
            (-0.007, -0.03, [0.048,0.029,0.022], [0.0090,0.0080,0.0068],
             [math.radians(22), math.radians(70), math.radians(42)]),
            ( 0.008,  0.03, [0.044,0.027,0.020], [0.0082,0.0074,0.0063],
             [math.radians(24), math.radians(72), math.radians(44)]),
            ( 0.023,  0.10, [0.033,0.020,0.015], [0.0066,0.0058,0.0050],
             [math.radians(28), math.radians(78), math.radians(48)]),
        ]
        for xf, xs, lengths, radii, curls in finger_defs:
            parts.extend(make_curled_finger(np.array([xf, y_mcp, 0.001]), xs, lengths, radii, curls))
        t_mcp = np.array([-0.044, -0.080, 0.012])
        parts.extend(make_curled_finger(
            mcp_pos=t_mcp, x_spread=-math.radians(30),
            lengths=[0.034, 0.026], radii=[0.0114, 0.0095],
            curl_angles=[math.radians(20), math.radians(45)], sections=12))
        return trimesh.util.concatenate(parts)

    # MONKEY-PATCH 1 — mesh builder
    pipe._build_test_arm_mesh = _build_realistic_arm_mesh

    # MONKEY-PATCH 2 — renderer: yellow skin, white background
    def _render_yellow(mesh, angle_deg, W, H, cam_dist, focal, bg_color):
        angle   = np.radians(angle_deg)
        cam_pos = np.array([cam_dist*np.cos(angle), 0, cam_dist*np.sin(angle)])
        forward = -cam_pos / np.linalg.norm(cam_pos)
        up      = np.array([0., 1., 0.])
        right   = np.cross(forward, up); right /= (np.linalg.norm(right) + 1e-8)
        cam_up  = np.cross(right, forward)
        R_mat   = np.array([right, -cam_up, forward])
        vc      = (R_mat @ (mesh.vertices*1000 - cam_pos).T).T
        z       = vc[:,2]; safe_z = np.maximum(z, 0.1)
        px      = (focal*vc[:,0]/safe_z + W/2.0).astype(int)
        py      = (focal*vc[:,1]/safe_z + H/2.0).astype(int)
        img     = np.full((H, W, 3), 255, dtype=np.uint8)
        light   = R_mat @ forward
        ndotl   = np.clip(mesh.face_normals @ light, 0, 1)
        skin    = np.array([230, 190, 40], dtype=np.float32)
        colors  = np.clip(skin*(0.25 + 0.75*ndotl[:,None]), 0, 255).astype(np.uint8)
        for fi, face in enumerate(mesh.faces):
            pts = np.array([[px[face[0]],py[face[0]]],[px[face[1]],py[face[1]]],
                            [px[face[2]],py[face[2]]]], dtype=np.int32)
            if z[face[0]]>1 and z[face[1]]>1 and z[face[2]]>1:
                cv2.fillConvexPoly(img, pts, colors[fi].tolist(), lineType=cv2.LINE_AA)
        return cv2.GaussianBlur(img, (3,3), 0.5)
    pipe._render_mesh = _render_yellow

    # MONKEY-PATCH 3 — generator: white background
    def _gen_white_bg(config):
        pipe.log.info('SYNTHETIC: Generating realistic fist images...')
        img_dir = config['image_dir']
        os.makedirs(img_dir, exist_ok=True)
        n  = config['num_images']
        W, H = 800, 600
        mesh   = pipe._build_test_arm_mesh()
        angles = np.linspace(config['angle_start_deg'], config['angle_end_deg'], n)
        bg = np.full((H, W, 3), 255, dtype=np.uint8)
        cv2.imwrite(os.path.join(img_dir, '_background.png'), bg)
        for i, angle in enumerate(angles):
            img = pipe._render_mesh(mesh, angle, W, H,
                                    config['camera_distance_mm'], W*1.2, (255,255,255))
            cv2.imwrite(os.path.join(img_dir, 'arm_{:03d}.png'.format(i)), img)
        pipe.log.info('  {} images saved.'.format(n))
        config['background_image'] = os.path.join(img_dir, '_background.png')
    pipe.generate_synthetic_test_data = _gen_white_bg

    # MONKEY-PATCH 4 — config overrides
    pipe.CONFIG['num_images']      = 72
    pipe.CONFIG['voxel_grid_size'] = 300

    # Preview the fist mesh
    new_mesh = _build_realistic_arm_mesh()
    bb = new_mesh.bounding_box.extents * 1000
    print('Mesh: {:,}V / {:,}F   BBox: {:.0f}x{:.0f}x{:.0f} mm'.format(
        len(new_mesh.vertices), len(new_mesh.faces), bb[0], bb[1], bb[2]))
    vP = new_mesh.vertices * 1000; fP = new_mesh.faces
    fig = plt.figure(figsize=(20, 7), facecolor='#0F172A')
    fig.suptitle('Relaxed Fist Preview — {:,}V / {:,}F'.format(len(vP), len(fP)),
                 color='white', fontsize=13, fontweight='bold')
    for i,(el,az,ttl) in enumerate([(20,50,'Perspective'),(0,0,'Front (XY)'),(90,0,'Top (XZ)')]):
        ax = fig.add_subplot(1,3,i+1, projection='3d')
        sub = fP[::max(1, len(fP)//3000)]
        p   = Poly3DCollection([vP[f] for f in sub], alpha=0.65, linewidth=0.04)
        p.set_facecolor('#E6BE28'); p.set_edgecolor('#92400E')
        ax.add_collection3d(p)
        ax.set_xlim(vP[:,0].min(),vP[:,0].max()); ax.set_ylim(vP[:,1].min(),vP[:,1].max())
        ax.set_zlim(vP[:,2].min(),vP[:,2].max())
        ax.view_init(elev=el, azim=az); ax.set_facecolor('#0F172A')
        ax.set_title(ttl, color='white', fontsize=10, pad=6)
        ax.tick_params(colors='#475569', labelsize=7)
        ax.set_xlabel('X mm', fontsize=8); ax.set_ylabel('Y mm', fontsize=8)
        ax.set_zlabel('Z mm', fontsize=8)
    plt.tight_layout()
    save_fig(STAGE_DIRS['S01'] + '/s00_fist_preview.png')
    print('Patches applied. num_images={}  voxel_grid_size={}'.format(
        pipe.CONFIG['num_images'], pipe.CONFIG['voxel_grid_size']))
    print('Preview saved. Run Stage 1 next.')

else:
    print(f'Real capture mode ({CAPTURE_MODE}) — synthetic patches skipped.')
    print(f'num_images={pipe.CONFIG["num_images"]}  '
          f'voxel_grid_size={pipe.CONFIG["voxel_grid_size"]}')

---
## Real Frame Loader
**Run for `real_rgb` or `real_rgbd` modes only.**  
This cell loads all frames from `real_capture/` into memory and copies images to `inputs/`.


In [ ]:
if CAPTURE_MODE != 'synthetic':
    print(f'Loading real frames from {CAPTURE_DIR}/rgb/ ...')

    rgb_paths = sorted(_g.glob(os.path.join(CAPTURE_DIR, 'rgb', 'frame_*.png')))
    assert len(rgb_paths) > 0, f'No frame_*.png files in {CAPTURE_DIR}/rgb/'

    # ── Copy RGB frames into INPUTS so pipeline functions can find them ───
    for p in rgb_paths:
        fname = os.path.basename(p).replace('frame_', 'arm_')
        shutil.copy(p, os.path.join(INPUTS, fname))

    # ── Copy or create background ─────────────────────────────────────────
    bg_src = os.path.join(CAPTURE_DIR, BG_FILENAME)
    if os.path.exists(bg_src):
        shutil.copy(bg_src, pipe.CONFIG['background_image'])
        print(f'  background.png copied')
    else:
        tmp_bgr = cv2.imread(rgb_paths[0])
        H0, W0  = tmp_bgr.shape[:2]
        cv2.imwrite(pipe.CONFIG['background_image'],
                    np.full((H0, W0, 3), 200, dtype=np.uint8))
        print('  WARNING: No background.png found. Using grey placeholder.')
        print('  Tip: Run g18_capture.py again — Step 0 captures the background.')

    # ── Load depth arrays for real_rgbd ──────────────────────────────────
    if CAPTURE_MODE == 'real_rgbd':
        depth_paths = sorted(_g.glob(os.path.join(CAPTURE_DIR, 'depth', 'frame_*.npy')))
        real_depths = [np.load(dp).astype(np.float32) for dp in depth_paths]
        print(f'  Depth arrays loaded: {len(real_depths)}')
        print(f'  Depth range sample: '
              f'{real_depths[0][real_depths[0]>0].min()*1000:.0f}mm – '
              f'{real_depths[0][real_depths[0]>0].max()*1000:.0f}mm')
    else:
        real_depths = []

    print(f'  {len(rgb_paths)} RGB frames loaded')

else:
    real_depths = []
    print('Synthetic mode — real frame loader skipped.')

---
## Stage 1 — Input Images
**Synthetic:** generates 72 arm photos 0–175°.  
**Real modes:** loads captured frames from `real_capture/rgb/`.


In [ ]:
if CAPTURE_MODE == 'synthetic':
    print('STAGE 1 — Generating synthetic images...')
    pipe.generate_synthetic_test_data(pipe.CONFIG)
    img_paths  = sorted(_g.glob(INPUTS + '/arm_*.png'))
    images     = [cv2.imread(p) for p in img_paths]
    images_rgb = [cv2.cvtColor(i, cv2.COLOR_BGR2RGB) for i in images]
    angles     = np.linspace(0, 175, len(images))
    stage1_label = 'Synthetic (Relaxed Fist, 72 views)'
else:
    print(f'STAGE 1 — Loading real images (CAPTURE_MODE={CAPTURE_MODE!r})...')
    img_paths  = sorted(_g.glob(INPUTS + '/arm_*.png'))
    images     = [cv2.imread(p) for p in img_paths]
    images_rgb = [cv2.cvtColor(i, cv2.COLOR_BGR2RGB) for i in images]
    angles     = np.linspace(0, 360, len(images), endpoint=False)
    stage1_label = f'Real capture ({CAPTURE_MODE}, {len(images)} frames)'

n_img = len(images)
print(f'  {n_img} images  {images[0].shape[1]}x{images[0].shape[0]} px')

# Grid display — adapt layout to frame count
cols = 12
rows_grid = max(1, (n_img + cols - 1) // cols)
fig, axes = plt.subplots(rows_grid, cols, figsize=(26, rows_grid*3+2),
                         facecolor='#0F172A')
fig.suptitle(f'STAGE 1 — {n_img} Input Images ({stage1_label})',
             color='white', fontsize=14, fontweight='bold')
axes_flat = axes.flat if rows_grid > 1 else axes
for idx, ax in enumerate(axes_flat):
    ax.set_facecolor('#0F172A')
    if idx < n_img:
        ax.imshow(images_rgb[idx])
        ax.set_title(f'{angles[idx]:.0f}deg', color='#94A3B8', fontsize=8, pad=2)
    ax.axis('off')
plt.tight_layout(pad=0.4)
save_fig(STAGE_DIRS['S01'] + '/s01_all_photos.png')

bg = cv2.cvtColor(cv2.imread(pipe.CONFIG['background_image']), cv2.COLOR_BGR2RGB)
fig, axes2 = plt.subplots(1, 2, figsize=(12, 4), facecolor='#0F172A')
fig.suptitle('Background vs Sample Photo', color='white', fontsize=12)
mid_idx = n_img // 2
axes2[0].imshow(bg);                axes2[0].set_title('Background', color='#94A3B8'); axes2[0].axis('off')
axes2[1].imshow(images_rgb[mid_idx]); axes2[1].set_title(f'Frame {mid_idx}', color='#94A3B8'); axes2[1].axis('off')
plt.tight_layout()
save_fig(STAGE_DIRS['S01'] + '/s01_bg_vs_arm.png')
print('Stage 1 done.')

---
## Stage 2 — Silhouette Extraction
**Synthetic / real_rgb:** multichannel background subtraction → edge morphology → GrabCut.  
**real_rgbd:** depth threshold (10–90 cm) → morphological cleanup. Much cleaner and faster.


In [ ]:
print('STAGE 2 — Extracting silhouettes...')

if CAPTURE_MODE == 'real_rgbd':
    # ── Depth-threshold silhouettes (main advantage of having depth data) ──
    print('  Using depth-threshold silhouettes (real_rgbd mode)...')
    ARM_NEAR_M = 0.10   # ignore objects closer than 10 cm
    ARM_FAR_M  = 0.90   # ignore background farther than 90 cm
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))

    silhouettes = []
    for depth_m in real_depths:
        mask = ((depth_m > ARM_NEAR_M) & (depth_m < ARM_FAR_M)).astype(np.uint8) * 255
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel, iterations=1)
        silhouettes.append(mask)
    print(f'  {len(silhouettes)} depth-threshold masks extracted')

else:
    # ── Colour-based silhouettes (synthetic or real_rgb) ─────────────────
    print('  Using colour-based silhouettes...')
    silhouettes = pipe.extract_silhouettes_enhanced(images, pipe.CONFIG)

fg_pct = [np.sum(s > 0)/s.size*100 for s in silhouettes]
print(f'  {len(silhouettes)} masks  avg_fg={np.mean(fg_pct):.1f}%')

selected = [int(i) for i in np.linspace(0, len(silhouettes)-1, 8, dtype=int)]
fig, axes = plt.subplots(2, 8, figsize=(22, 6), facecolor='#0F172A')
fig.suptitle('STAGE 2 — Photo (top) vs Silhouette Mask (bottom)',
             color='white', fontsize=13, fontweight='bold')
for col, idx in enumerate(selected):
    axes[0,col].imshow(images_rgb[idx])
    axes[0,col].set_title(f'{angles[idx]:.0f}deg', color='#93C5FD', fontsize=9)
    axes[0,col].axis('off')
    axes[1,col].imshow(silhouettes[idx], cmap='gray', vmin=0, vmax=255)
    axes[1,col].set_title(f'fg={fg_pct[idx]:.0f}%', color='#6EE7B7', fontsize=9)
    axes[1,col].axis('off')
plt.tight_layout(pad=0.3)
save_fig(STAGE_DIRS['S02'] + '/s02_photo_vs_mask.png')

fig, ax = plt.subplots(figsize=(16,4), facecolor='#0F172A')
ax.bar(range(len(fg_pct)), fg_pct, color='#3B82F6', alpha=0.85, width=0.8)
ax.axhline(np.mean(fg_pct), color='#F59E0B', ls='--', lw=1.5,
           label=f'Mean {np.mean(fg_pct):.1f}%')
ax.set_xlabel('Image index'); ax.set_ylabel('Foreground %')
ax.set_title('Foreground Coverage per View', color='white', fontsize=12)
ax.legend(facecolor='#1E293B', labelcolor='white')
plt.tight_layout()
save_fig(STAGE_DIRS['S02'] + '/s02_fg_coverage.png')
print('Stage 2 done.')

---
## Stage 3 — Visual Hull Voxel Carving
300³ grid starts fully occupied. Each silhouette view carves voxels outside the mask.  
Works for all three capture modes — depth-threshold silhouettes give cleaner carving in `real_rgbd`.


In [ ]:
print('STAGE 3 — Voxel carving...')
intrinsics_vc = pipe.init_rgb_camera(pipe.CONFIG, images)
projections   = pipe.compute_camera_matrices_rgb(intrinsics_vc, len(images), pipe.CONFIG)
G = pipe.CONFIG['voxel_grid_size']
vol_mm = pipe.CONFIG['voxel_volume_mm']
voxel_sz = vol_mm / G
half = vol_mm / 2.0
xs   = np.linspace(-half+voxel_sz/2, half-voxel_sz/2, G)
grid = np.stack(np.meshgrid(xs,xs,xs,indexing='ij'),axis=-1).reshape(-1,3)
occupied = np.ones(len(grid), dtype=bool)
N0 = occupied.sum()
vlog = [N0]
for sil,proj in zip(silhouettes, projections):
    ph = np.hstack([grid[occupied], np.ones((occupied.sum(),1))])
    pp = (proj @ ph.T).T
    sz2 = np.maximum(pp[:,2],1e-6)
    u,v = (pp[:,0]/sz2).astype(int),(pp[:,1]/sz2).astype(int)
    hs,ws = sil.shape
    inb = (u>=0)&(u<ws)&(v>=0)&(v<hs)&(pp[:,2]>0)
    on = np.zeros(occupied.sum(),dtype=bool)
    vi = np.where(inb)[0]; on[vi] = sil[v[vi],u[vi]]>128
    ai = np.where(occupied)[0]; occupied[ai[~on]] = False
    vlog.append(occupied.sum())
occupancy = occupied.reshape(G,G,G)
print('  Final: {:,} voxels ({:.1f}%)'.format(occupied.sum(), occupied.sum()/N0*100))

pct2 = [v/N0*100 for v in vlog]
fig, ax = plt.subplots(figsize=(16,4), facecolor='#0F172A')
ax.plot(pct2, color='#3B82F6', lw=2)
ax.fill_between(range(len(pct2)), pct2, alpha=0.25, color='#3B82F6')
ax.annotate('Final: {:.1f}%'.format(pct2[-1]), xy=(len(pct2)-1,pct2[-1]),
            color='#F59E0B', xytext=(-90,25), textcoords='offset points',
            arrowprops=dict(arrowstyle='->', color='#F59E0B'))
ax.set_xlabel('View index'); ax.set_ylabel('Occupied voxels %')
ax.set_title('STAGE 3 — Voxels Remaining After Each View', color='white', fontsize=12)
plt.tight_layout()
save_fig(STAGE_DIRS['S03'] + '/s03_carving.png')

mid = G//2
fig, axes = plt.subplots(1,3, figsize=(18,6), facecolor='#0F172A')
fig.suptitle('STAGE 3 — Occupancy Grid: Three Slices', color='white', fontsize=13)
for ax,(slc,ttl,xl,yl) in zip(axes,[
    (occupancy[:,:,mid],'XY top','X','Y'),(occupancy[:,mid,:],'XZ front','X','Z'),
    (occupancy[mid,:,:],'YZ side','Y','Z')]):
    ax.imshow(slc.T,origin='lower',cmap='Blues',extent=[-half,half,-half,half])
    ax.set_title(ttl,color='white',fontsize=11)
    ax.set_xlabel(xl+' (mm)',color='#94A3B8'); ax.set_ylabel(yl+' (mm)',color='#94A3B8')
    ax.axhline(0,color='#475569',lw=0.5,ls='--'); ax.axvline(0,color='#475569',lw=0.5,ls='--')
plt.tight_layout()
save_fig(STAGE_DIRS['S03'] + '/s03_slices.png')
print('Stage 3 done.')

---
## Stage 4 — Marching Cubes Mesh
Gaussian sigma=1.0 then Marching Cubes at iso-level 0.5 extracts the triangulated surface.

In [ ]:
print('STAGE 4 — Marching Cubes...')
smooth = ndimage.gaussian_filter(occupancy.astype(np.float32), sigma=1.0)
vM,fM,_,_ = measure.marching_cubes(smooth, level=pipe.CONFIG['marching_cubes_level'])
vMm = vM * voxel_sz - half
hand_mesh_raw = trimesh.Trimesh(vertices=vMm/1000.0, faces=fM, process=True)
print('  Vertices: {:,}   Faces: {:,}'.format(len(vMm), len(fM)))
fig = plt.figure(figsize=(18,6), facecolor='#0F172A')
fig.suptitle('STAGE 4 — Marching Cubes Mesh  ({:,}V / {:,}F)'.format(len(vMm),len(fM)),
             color='white', fontsize=13, fontweight='bold')
for i,(el,az,ttl) in enumerate([(20,45,'Perspective'),(90,0,'Top'),(0,0,'Front')]):
    ax = fig.add_subplot(1,3,i+1, projection='3d')
    plot_mesh_3d(ax, vMm, fM, ttl, elev=el, azim=az)
plt.tight_layout()
save_fig(STAGE_DIRS['S04'] + '/s04_mesh.png')
fig, axes = plt.subplots(1,2, figsize=(14,5), facecolor='#0F172A')
fig.suptitle('STAGE 4 — Smoothing Effect on Marching Cubes Input', color='white', fontsize=12)
for ax,slc,ttl in zip(axes,
    [occupancy[:,:,mid].astype(float), smooth[:,:,mid]],
    ['Raw occupancy (blocky)','After Gaussian sigma=1.0']):
    ax.imshow(slc.T,origin='lower',cmap='Blues',extent=[-half,half,-half,half])
    if slc.max()>0.5:
        ax.contour(np.linspace(-half,half,G),np.linspace(-half,half,G),
                   slc.T,levels=[0.5],colors=['#F59E0B'],linewidths=2)
    ax.set_title(ttl,color='white',fontsize=11)
    ax.set_xlabel('X (mm)',color='#94A3B8'); ax.set_ylabel('Z (mm)',color='#94A3B8')
plt.tight_layout()
save_fig(STAGE_DIRS['S04'] + '/s04_smooth.png')
print('Stage 4 done.')

---
## Stage 5 — Convex Hull Closure
Fills unseen underside. Guarantees watertight mesh with Euler number 2.

In [ ]:
print('STAGE 5 — Convex hull...')
hand_hull = pipe.close_with_convex_hull(hand_mesh_raw, pipe.CONFIG)
vO = hand_mesh_raw.vertices * 1000; vH = hand_hull.vertices * 1000
print('  Original: {:,}V  WT={}'.format(len(vO), hand_mesh_raw.is_watertight))
print('  Hull:     {:,}V  WT={}  Euler={}'.format(len(vH), hand_hull.is_watertight, hand_hull.euler_number))
fig = plt.figure(figsize=(20,7), facecolor='#0F172A')
fig.suptitle('STAGE 5 — Convex Hull Closure', color='white', fontsize=13, fontweight='bold')
ax1 = fig.add_subplot(1,3,1,projection='3d')
plot_mesh_3d(ax1, vO, hand_mesh_raw.faces,
             'Original (open)\n{:,}V WT={}'.format(len(vO), hand_mesh_raw.is_watertight),
             elev=15, azim=60, color='#3B82F6', alpha=0.5)
ax2 = fig.add_subplot(1,3,2,projection='3d')
plot_mesh_3d(ax2, vH, hand_hull.faces,
             'Hull (closed)\n{:,}V Euler={}'.format(len(vH), hand_hull.euler_number),
             elev=15, azim=60, color='#10B981', alpha=0.5)
ax3 = fig.add_subplot(1,3,3,projection='3d')
so = hand_mesh_raw.faces[::max(1,len(hand_mesh_raw.faces)//1500)]
sh = hand_hull.faces[::max(1,len(hand_hull.faces)//1500)]
po = Poly3DCollection([vO[f] for f in so],alpha=0.2,linewidth=0.05)
po.set_facecolor('#3B82F6'); po.set_edgecolor('#1E40AF')
ph2 = Poly3DCollection([vH[f] for f in sh],alpha=0.15,linewidth=0.4)
ph2.set_facecolor('#10B981'); ph2.set_edgecolor('#059669')
ax3.add_collection3d(po); ax3.add_collection3d(ph2)
av = np.vstack([vO,vH])
ax3.set_xlim(av[:,0].min(),av[:,0].max()); ax3.set_ylim(av[:,1].min(),av[:,1].max())
ax3.set_zlim(av[:,2].min(),av[:,2].max()); ax3.view_init(15,60); ax3.set_facecolor('#0F172A')
ax3.set_title('Overlay', color='white', fontsize=10)
ax3.legend(handles=[
    mpatches.Patch(color='#3B82F6',alpha=0.7,label='Original'),
    mpatches.Patch(color='#10B981',alpha=0.7,label='Hull')],
    facecolor='#1E293B',labelcolor='white',fontsize=9)
plt.tight_layout()
save_fig(STAGE_DIRS['S05'] + '/s05_hull.png')
print('Stage 5 done.')

---
## Stage 6 — Mesh Cleaning
fill_holes, fix_winding, fix_normals, remove_degenerates, Laplacian smooth 30 iter.

In [ ]:
print('STAGE 6 — Mesh cleaning...')
before_mesh = copy.deepcopy(hand_hull)
hand_clean  = pipe.clean_and_validate(hand_hull, pipe.CONFIG, label='hand_cleaned')
vBef = before_mesh.vertices * 1000; vCln = hand_clean.vertices * 1000
print('  Before: {:,}V WT={} Euler={}'.format(len(vBef),before_mesh.is_watertight,before_mesh.euler_number))
print('  After:  {:,}V WT={} Euler={}'.format(len(vCln),hand_clean.is_watertight,hand_clean.euler_number))
fig = plt.figure(figsize=(16,6), facecolor='#0F172A')
fig.suptitle('STAGE 6 — Mesh Cleaning: Before vs After', color='white', fontsize=13)
for col,(V,F,ttl,clr) in enumerate([
    (vBef,before_mesh.faces,'Before\n{:,}V Euler={}'.format(len(vBef),before_mesh.euler_number),'#F59E0B'),
    (vCln,hand_clean.faces, 'After\n{:,}V Euler={}'.format(len(vCln),hand_clean.euler_number), '#10B981')]):
    ax = fig.add_subplot(1,2,col+1,projection='3d')
    plot_mesh_3d(ax, V, F, ttl, elev=20, azim=50, color=clr, alpha=0.5)
plt.tight_layout()
save_fig(STAGE_DIRS['S06'] + '/s06_clean.png')
bb = hand_clean.bounding_box.extents * 1000
fig, ax = plt.subplots(figsize=(10,4), facecolor='#0F172A'); ax.axis('off')
rows = [['Metric','Before','After','Target'],
    ['Vertices','{:,}'.format(len(vBef)),'{:,}'.format(len(vCln)),'—'],
    ['Faces','{:,}'.format(len(before_mesh.faces)),'{:,}'.format(len(hand_clean.faces)),'—'],
    ['Watertight',str(before_mesh.is_watertight),str(hand_clean.is_watertight),'True'],
    ['Euler',str(before_mesh.euler_number),str(hand_clean.euler_number),'2'],
    ['BBox X','—','{:.1f}mm'.format(bb[0]),'~90mm'],
    ['BBox Y','—','{:.1f}mm'.format(bb[1]),'~320mm'],
    ['BBox Z','—','{:.1f}mm'.format(bb[2]),'~90mm']]
t = ax.table(cellText=rows[1:],colLabels=rows[0],cellLoc='center',loc='center',bbox=[0,0,1,1])
t.auto_set_font_size(False); t.set_fontsize(12)
for (r,c),cell in t.get_celld().items():
    cell.set_facecolor('#1E3A5F' if r==0 else ('#1E293B' if r%2==0 else '#0F172A'))
    cell.set_text_props(color='white'); cell.set_edgecolor('#334155')
ax.set_title('Mesh Quality Metrics', color='white', fontsize=12, pad=10)
plt.tight_layout()
save_fig(STAGE_DIRS['S06'] + '/s06_stats.png')
print('Stage 6 done.')

---
## Stage 7 — Boolean Cast Shell and Split
Plus 3mm inner offset, plus 7mm outer offset, Boolean difference = hollow shell. Then Voronoi and split.

In [ ]:
print('STAGE 7 — Generating cast...')
gap  = pipe.CONFIG['cast_gap_mm']  / 1000.0
wall = pipe.CONFIG['cast_wall_mm'] / 1000.0
inner_verts = hand_clean.vertices + hand_clean.vertex_normals * gap
outer_verts = hand_clean.vertices + hand_clean.vertex_normals * (gap+wall)
top_shell, bottom_shell, full_cast = pipe.generate_cast_design(hand_clean, pipe.CONFIG)
vH2  = hand_clean.vertices * 1000; vInn = inner_verts * 1000; vOut = outer_verts * 1000
vFC  = full_cast.vertices * 1000;  vTop = top_shell.vertices * 1000
vBot = bottom_shell.vertices * 1000
print('  Cast: {:,}F  WT={}'.format(len(full_cast.faces), full_cast.is_watertight))
print('  Top:  {:,}F  Bottom: {:,}F'.format(len(top_shell.faces), len(bottom_shell.faces)))
fig, axes = plt.subplots(1,2, figsize=(16,7), facecolor='#0F172A')
fig.suptitle('STAGE 7 — Boolean Cast Shell', color='white', fontsize=13, fontweight='bold')
axl = axes[0]; axl.set_facecolor('#1E293B')
cx3 = np.median(vH2[:,0]); tol3=(vH2[:,0].max()-vH2[:,0].min())*0.06
for V,clr,lbl,sz3 in [(vH2,'#FDE68A','Arm',6),(vInn,'#B91C1C','+3mm',4),(vOut,'#3B82F6','+7mm',4)]:
    mk = np.abs(V[:,0]-cx3)<tol3
    if mk.sum()>0: axl.scatter(V[mk,1],V[mk,2],c=clr,s=sz3,alpha=0.7,label=lbl)
axl.set_xlabel('Y (mm)',color='white'); axl.set_ylabel('Z (mm)',color='white')
axl.set_title('YZ Cross-Section: Offset Layers',color='white')
axl.legend(facecolor='#0F172A',labelcolor='white',fontsize=9); axl.set_aspect('equal')
axr = fig.add_subplot(1,2,2,projection='3d')
plot_mesh_3d(axr,vFC,full_cast.faces,
             'Full Cast\n{:,}V / {:,}F'.format(len(vFC),len(full_cast.faces)),
             elev=20,azim=50,color='#3B82F6',alpha=0.45)
plt.tight_layout()
save_fig(STAGE_DIRS['S07'] + '/s07_shell.png')
from matplotlib.patches import Ellipse
fig, ax = plt.subplots(figsize=(9,6),facecolor='#0F172A')
ax.set_facecolor('#1E293B'); ax.set_xlim(-130,130); ax.set_ylim(-100,110); ax.set_aspect('equal')
ax.set_title('Cast Shell Cross-Section Anatomy',color='white',fontsize=13)
for rx,ry,fc,ec,al in [(103,78,'#3B82F6','#3B82F6',0.25),(85,63,'#1E293B','#B91C1C',0.0),(75,55,'#FDE68A','#D97706',0.9)]:
    ax.add_patch(Ellipse((0,0),rx*2,ry*2,facecolor=fc,edgecolor=ec,alpha=al,linewidth=2))
ax.text(0,0,'ARM',ha='center',va='center',color='#92400E',fontsize=14,fontweight='bold')
ax.annotate('',xy=(85,0),xytext=(75,0),arrowprops=dict(arrowstyle='<->',color='#B91C1C',lw=2))
ax.text(80,7,'3mm gap',ha='center',color='#FCA5A5',fontsize=9)
ax.annotate('',xy=(103,0),xytext=(85,0),arrowprops=dict(arrowstyle='<->',color='#60A5FA',lw=2))
ax.text(94,7,'4mm wall',ha='center',color='#93C5FD',fontsize=9)
for hx,hy in [(-100,30),(-95,-40),(90,20),(98,-35)]:
    ax.add_patch(plt.Circle((hx,hy),8,color='#1E293B',zorder=5))
    ax.add_patch(plt.Circle((hx,hy),8,fill=False,edgecolor='#7C3AED',lw=1.5,zorder=6))
ax.text(-115,55,'Voronoi\nholes',color='#A78BFA',fontsize=9,ha='center')
ax.axis('off'); plt.tight_layout()
save_fig(STAGE_DIRS['S07'] + '/s07_xsec.png')
print('Stage 7 done.')

---
## Stage 8 — Voronoi Hole Analysis
Visualise 150 hole centres on the cast surface before Boolean subtraction.

In [ ]:
print('STAGE 8 — Voronoi analysis...')
n_pts3 = pipe.CONFIG['voronoi_num_points']
rad3   = pipe.CONFIG['voronoi_hole_radius'] / 1000.0
pts3,_ = trimesh.sample.sample_surface(full_cast, n_pts3)
dist3  = np.linalg.norm(pts3 - full_cast.centroid, axis=1)
val3   = dist3 < max(full_cast.bounding_box.extents)*0.45
hc3    = pts3[val3]; hcm3 = hc3*1000
print('  {} holes  r={:.0f}mm'.format(len(hc3), rad3*1000))
fig = plt.figure(figsize=(20,7),facecolor='#0F172A')
fig.suptitle('STAGE 8 — Voronoi Hole Positions ({} holes, r={:.0f}mm)'.format(len(hc3),rad3*1000),
             color='white',fontsize=13,fontweight='bold')
for i,(el,az,ttl) in enumerate([(20,45,'Perspective'),(90,0,'Top'),(0,90,'Side')]):
    ax = fig.add_subplot(1,3,i+1,projection='3d')
    sub = full_cast.faces[::max(1,len(full_cast.faces)//1500)]
    pp3 = Poly3DCollection([vFC[f] for f in sub],alpha=0.15,linewidth=0.05)
    pp3.set_facecolor('#3B82F6'); pp3.set_edgecolor('#1E40AF')
    ax.add_collection3d(pp3)
    ax.scatter(hcm3[:,0],hcm3[:,1],hcm3[:,2],c='#EF4444',s=30,alpha=0.9,zorder=5)
    ax.set_xlim(vFC[:,0].min(),vFC[:,0].max()); ax.set_ylim(vFC[:,1].min(),vFC[:,1].max())
    ax.set_zlim(vFC[:,2].min(),vFC[:,2].max())
    ax.view_init(el,az); ax.set_facecolor('#0F172A')
    ax.set_title(ttl,color='white',fontsize=10); ax.tick_params(colors='#475569',labelsize=7)
fig.legend(handles=[mpatches.Patch(color='#3B82F6',alpha=0.6,label='Cast'),
    mpatches.Patch(color='#EF4444',label='Holes')],
    facecolor='#1E293B',labelcolor='white',fontsize=10,loc='lower center',ncol=2)
plt.tight_layout(rect=[0,0.05,1,1])
save_fig(STAGE_DIRS['S08'] + '/s08_positions.png')
fig, axes = plt.subplots(1,2,figsize=(14,5),facecolor='#0F172A')
axes[0].hist(dist3[val3]*1000,bins=20,color='#7C3AED',edgecolor='#0F172A',alpha=0.85)
axes[0].set_facecolor('#1E293B'); axes[0].set_title('Distance from centroid',color='white')
axes[0].set_xlabel('mm',color='white'); axes[0].set_ylabel('Count',color='white')
axes[1].scatter(hcm3[:,1],hcm3[:,2],c='#EF4444',s=20,alpha=0.7)
axes[1].set_facecolor('#1E293B'); axes[1].set_aspect('equal')
axes[1].set_title('YZ projection of hole centres',color='white')
axes[1].set_xlabel('Y (mm)',color='white'); axes[1].set_ylabel('Z (mm)',color='white')
plt.tight_layout()
save_fig(STAGE_DIRS['S08'] + '/s08_dist.png')
print('Stage 8 done.')

---
## Stage 9 — Clamshell Split
Split along X-axis at centroid with 0.5mm gap. Face majority vote assigns each face to a half.

In [ ]:
print('STAGE 9 — Clamshell split...')
print(f'  Top:    {len(top_shell.faces):,}F  WT={top_shell.is_watertight}')
print(f'  Bottom: {len(bottom_shell.faces):,}F  WT={bottom_shell.is_watertight}')
fig = plt.figure(figsize=(20,7), facecolor='#0F172A')
fig.suptitle('STAGE 9 — Clamshell Split: Full Cast to Two Printable Halves',
             color='white', fontsize=13, fontweight='bold')
for i,(V,F,clr,ttl) in enumerate([
    (vFC,  full_cast.faces,    '#6B7280', 'Full Cast\n{} faces'.format(len(full_cast.faces))),
    (vTop, top_shell.faces,    '#3B82F6', 'Top Shell\n{} faces'.format(len(top_shell.faces))),
    (vBot, bottom_shell.faces, '#EF4444', 'Bottom Shell\n{} faces'.format(len(bottom_shell.faces))),
]):
    ax = fig.add_subplot(1,3,i+1, projection='3d')
    plot_mesh_3d(ax, V, F, ttl, elev=20, azim=40, color=clr, alpha=0.5)
plt.tight_layout()
save_fig(STAGE_DIRS['S09'] + '/s09_split.png')
fig = plt.figure(figsize=(16,7), facecolor='#0F172A')
fig.suptitle('STAGE 9 — Assembled: Top (blue) + Bottom (red)', color='white', fontsize=13)
for i,(el,az) in enumerate([(15,30),(15,120)]):
    ax = fig.add_subplot(1,2,i+1, projection='3d')
    st = top_shell.faces[::max(1,len(top_shell.faces)//1200)]
    sb = bottom_shell.faces[::max(1,len(bottom_shell.faces)//1200)]
    pt = Poly3DCollection([vTop[f] for f in st], alpha=0.45, linewidth=0.05)
    pt.set_facecolor('#3B82F6'); pt.set_edgecolor('#1E40AF')
    pb = Poly3DCollection([vBot[f] for f in sb], alpha=0.45, linewidth=0.05)
    pb.set_facecolor('#EF4444'); pb.set_edgecolor('#7F1D1D')
    ax.add_collection3d(pt); ax.add_collection3d(pb)
    av = np.vstack([vTop,vBot])
    ax.set_xlim(av[:,0].min(),av[:,0].max())
    ax.set_ylim(av[:,1].min(),av[:,1].max())
    ax.set_zlim(av[:,2].min(),av[:,2].max())
    ax.view_init(el,az); ax.set_facecolor('#0F172A')
    ax.set_title('Assembled view {}'.format(i+1), color='white', fontsize=10)
    ax.tick_params(colors='#475569', labelsize=7)
    ax.legend(handles=[
        mpatches.Patch(color='#3B82F6',alpha=0.7,label='Top'),
        mpatches.Patch(color='#EF4444',alpha=0.7,label='Bottom')],
        facecolor='#1E293B', labelcolor='white', fontsize=9)
plt.tight_layout()
save_fig(STAGE_DIRS['S09'] + '/s09_assembled.png')
print('Stage 9 done.')

---
## Stage 10 — STL Export and Fit Validation
5000 points from cast measure distance to hand mesh.  
Mean should equal `cast_gap_mm = 3mm` for a correct fit.  
**Bug fix:** `fit_threshold_mm` corrected to `cast_gap_mm + 1.0 = 4.0`.


In [ ]:
import time as _t, shutil as _sh, glob as _gl
print('STAGE 10 — STL Export and Fit Validation...')

# ── BUG FIX: fit_threshold_mm should be cast_gap_mm + 1.0 ───────────────
pipe.CONFIG['fit_threshold_mm'] = pipe.CONFIG['cast_gap_mm'] + 1.0   # was 2.0, now 4.0
print(f'  fit_threshold_mm corrected → {pipe.CONFIG["fit_threshold_mm"]:.1f} mm')

t0 = _t.time()
report = pipe.export_and_report(top_shell, bottom_shell, full_cast, hand_clean,
                                 pipe.CONFIG, CAPTURE_MODE, _t.time()-t0)
for f in _gl.glob(pipe.CONFIG['mesh_dir'] + '/*.stl'):
    _sh.copy(f, STAGE_DIRS['S10'])

fit_fc = report['fit_validation']['full_cast']
fit_ts = report['fit_validation']['top_shell']
meshes = report['meshes']
print('  full_cast fit: mean={:.2f}mm  max={:.2f}mm'.format(
    fit_fc['mean_mm'], fit_fc['max_mm']))
print('  top_shell fit: mean={:.2f}mm'.format(fit_ts['mean_mm']))

metrics = ['mean_mm','std_mm','p90_mm','p95_mm','max_mm']
labels  = ['Mean','Std','P90','P95','Max']
x = np.arange(len(labels)); w = 0.35
fcv = [fit_fc.get(m,0) for m in metrics]
tsv = [fit_ts.get(m,0) for m in metrics]

fig, axes = plt.subplots(1,2, figsize=(16,5), facecolor='#0F172A')
fig.suptitle('STAGE 10 — Fit Validation and Mesh Quality', color='white', fontsize=13)
ax = axes[0]; ax.set_facecolor('#1E293B')
b1 = ax.bar(x-w/2, fcv, w, label='full_cast', color='#10B981', alpha=0.85)
b2 = ax.bar(x+w/2, tsv, w, label='top_shell', color='#EF4444', alpha=0.75)
ax.axhline(pipe.CONFIG['cast_gap_mm'], color='#F59E0B', ls='--', lw=1.5,
           label='Expected {:.0f}mm'.format(pipe.CONFIG['cast_gap_mm']))
ax.axhline(pipe.CONFIG['fit_threshold_mm'], color='#A78BFA', ls=':', lw=1.5,
           label='Threshold {:.0f}mm'.format(pipe.CONFIG['fit_threshold_mm']))
ax.set_xticks(x); ax.set_xticklabels(labels, color='white')
ax.set_ylabel('Deviation (mm)', color='white')
ax.set_title('Fit Deviation Metrics', color='white')
ax.legend(facecolor='#0F172A', labelcolor='white', fontsize=9)
for bar in b1:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
            '{:.1f}'.format(bar.get_height()), ha='center', color='#6EE7B7', fontsize=8)

ax2 = axes[1]; ax2.axis('off')
rows = []
for key, m in meshes.items():
    bb = m['bounding_box_mm']
    rows.append([key.replace('_',' ').title(),
                 '{:,}'.format(m['vertices']),
                 '{:,}'.format(m['faces']),
                 'YES' if m['watertight'] else 'NO',
                 str(m['euler_number']),
                 '{:.0f}x{:.0f}x{:.0f}'.format(bb[0],bb[1],bb[2])])
t = ax2.table(cellText=rows,
              colLabels=['Mesh','Verts','Faces','WT','Euler','BBox mm'],
              cellLoc='center', loc='center', bbox=[0,0,1,1])
t.auto_set_font_size(False); t.set_fontsize(10)
for (r,c),cell in t.get_celld().items():
    cell.set_facecolor('#1E3A5F' if r==0 else ('#1E293B' if r%2==0 else '#0F172A'))
    cell.set_text_props(color='white'); cell.set_edgecolor('#334155')
ax2.set_title('Output Mesh Quality', color='white', fontsize=11, pad=12)
plt.tight_layout()
save_fig(STAGE_DIRS['S10'] + '/s10_fit.png')

dr = report['defense_readiness']
fig, ax = plt.subplots(figsize=(10,4), facecolor='#0F172A')
ax.set_facecolor('#0F172A'); ax.axis('off')
ax.set_title('Defense Readiness Dashboard', color='white', fontsize=14,
             fontweight='bold', pad=20)
for i,(k,v) in enumerate(dr.items()):
    color = '#10B981' if v else '#EF4444'
    icon  = 'PASS' if v else 'FAIL'
    xp = (i%2)*0.5+0.1; yp = 0.75-(i//2)*0.35
    ax.text(xp, yp, '[{}]  {}'.format(icon, k.replace('_',' ').title()),
            transform=ax.transAxes, color=color, fontsize=13,
            bbox=dict(boxstyle='round,pad=0.4', facecolor='#1E293B',
                      edgecolor=color, linewidth=1.5))
plt.tight_layout()
save_fig(STAGE_DIRS['S10'] + '/s10_readiness.png')

stls = _gl.glob(STAGE_DIRS['S10'] + '/*.stl')
print('Stage 10 done.  STL files: {}'.format(len(stls)))
for f in stls:
    print('  {:35s} {:5d} KB'.format(os.path.basename(f), os.path.getsize(f)//1024))

---
## Final Cell — Generate HTML Visual Walkthrough
Assembles all stage output images into a single portable HTML file.

In [ ]:
import base64, time as _t2

def b64(path):
    if not os.path.exists(path): return ''
    with open(path,'rb') as f: return base64.b64encode(f.read()).decode()

# Update Stage 1 label in HTML based on capture mode
_s1_desc = {
    'synthetic': ('Input: Synthetic Arm Photos',
                  'The pipeline generates 72 synthetic arm photos covering 0 to 175 degrees, '
                  'simulating a turntable capture session. A clean background is also saved for '
                  'background subtraction.'),
    'real_rgb':  ('Input: Real RGB Photos',
                  'Real photographs captured with a standard camera. '
                  'Colour-based silhouette extraction (background subtraction + GrabCut).'),
    'real_rgbd': ('Input: Real RealSense D435 Frames',
                  'Intel RealSense D435 RGB+D frames captured with g18_capture.py. '
                  'Depth-threshold silhouettes replace colour-based extraction.'),
}.get(CAPTURE_MODE, ('Input Images', ''))

STAGES = [
    ('S01','01','#3B82F6', _s1_desc[0], _s1_desc[1],
     [('s01_all_photos.png','Input images grid'),
      ('s01_bg_vs_arm.png', 'Background reference vs sample frame')]),

    ('S02','02','#8B5CF6','Silhouette Extraction',
     'Depth-threshold (real_rgbd) or colour background subtraction + GrabCut (other modes). '
     'Output: binary mask — white = arm, black = background.',
     [('s02_photo_vs_mask.png','Original photos (top) vs binary silhouette masks (bottom)'),
      ('s02_fg_coverage.png',  'Foreground coverage per view')]),

    ('S03','03','#06B6D4','Visual Hull Voxel Carving',
     'A 300-cubed voxel grid starts fully occupied. Each silhouette view carves away '
     'voxels projecting outside the mask.',
     [('s03_carving.png', 'Occupied voxels remaining after each view'),
      ('s03_slices.png',  'Three orthogonal slices through the final occupancy grid')]),

    ('S04','04','#10B981','Marching Cubes Mesh',
     'Gaussian smooth sigma=1.0 removes voxel staircase artifacts. '
     'Marching Cubes extracts a triangulated surface at iso-level 0.5.',
     [('s04_mesh.png',   'Raw mesh from three viewpoints'),
      ('s04_smooth.png', 'Effect of Gaussian smoothing on the input field')]),

    ('S05','05','#F59E0B','Convex Hull Closure',
     'Multi-view capture cannot see the arm underside. The convex hull fills this '
     'unseen geometry and guarantees a watertight mesh with Euler number 2.',
     [('s05_hull.png','Original open mesh vs convex hull vs overlay')]),

    ('S06','06','#EF4444','Mesh Cleaning',
     'Repair sequence: fill holes, fix winding, fix normals, remove degenerates, '
     'then Laplacian smoothing 30 iterations lambda=0.5.',
     [('s06_clean.png', 'Before vs after cleaning and Laplacian smoothing'),
      ('s06_stats.png', 'Quantitative mesh quality metrics before and after')]),

    ('S07','07','#6366F1','Boolean Cast Shell',
     'Vertex normals offset the mesh plus 3mm inner and plus 7mm outer. '
     'Boolean difference creates the hollow shell. Then Voronoi holes and fastener holes.',
     [('s07_shell.png', 'YZ cross-section showing offset layers and 3D cast mesh'),
      ('s07_xsec.png',  'Annotated diagram: 3mm gap, 4mm wall, Voronoi holes')]),

    ('S08','08','#D97706','Voronoi Ventilation Holes',
     '150 seed points sampled on the shell surface define hole centres after edge filtering. '
     'Each hole is a 5mm-radius sphere Boolean-subtracted from the shell.',
     [('s08_positions.png', 'Hole centre positions on the cast surface from three angles'),
      ('s08_dist.png',      'Distance histogram and YZ projection of hole positions')]),

    ('S09','09','#EC4899','Clamshell Split',
     'The cast is split along the X-axis at its centroid with a 0.5mm gap. '
     'Each half has fill_holes applied to close the cut boundary.',
     [('s09_split.png',    'Full cast split into top blue and bottom red printable shells'),
      ('s09_assembled.png','Both halves assembled together from two angles')]),

    ('S10','10','#059669','STL Export and Fit Validation',
     '5000 surface points from the cast measure distance to the hand mesh. '
     'Mean deviation should equal cast_gap_mm = 3mm for a correct fit. '
     'fit_threshold_mm corrected to 4.0.',
     [('s10_fit.png',      'Fit deviation metrics and mesh quality summary'),
      ('s10_readiness.png','Defense readiness checklist')]),
]

def make_card(sid, snum, color, title, desc, images, d):
    imgs = ''
    for fname, cap in images:
        path = d + '/' + fname
        data = b64(path)
        if data:
            imgs += ('<figure class="sf">'
                     '<img src="data:image/png;base64,{}" alt="{}">'
                     '<figcaption>{}</figcaption></figure>'.format(data, cap, cap))
        else:
            imgs += '<p class="miss">Image not found: {}</p>'.format(fname)
    return (
        '<section class="sc" id="{0}">'
        '<div class="sh" style="border-left:6px solid {2}">'
        '<span class="sn" style="background:{2}">STAGE {1}</span>'
        '<h2>{3}</h2></div>'
        '<p class="sd">{4}</p>'
        '<div class="si">{5}</div>'
        '</section>'
    ).format(sid, snum, color, title, desc, imgs)

def get_dir(sid):
    for k,v in STAGE_DIRS.items():
        if k == sid: return v
    return ''

cards = ''
for s in STAGES:
    cards += make_card(s[0], s[1], s[2], s[3], s[4], s[5], get_dir(s[0]))

toc  = ''.join('<a href="#{}">{} {}</a>'.format(s[0],s[1],s[3]) for s in STAGES)
flow = '<span class="fa">&#8594;</span>'.join(
       '<a class="fs" href="#{}">{} {}</a>'.format(s[0],s[1],s[3]) for s in STAGES)

CSS = (
    ':root{--bg:#0F172A;--card:#1E293B;--bd:#334155;--tx:#E2E8F0;--mt:#94A3B8;--cp:#64748B}'
    '*{box-sizing:border-box;margin:0;padding:0}'
    'body{background:var(--bg);color:var(--tx);font-family:"Segoe UI",Arial,sans-serif;'
    'font-size:15px;line-height:1.7}'
    '.hd{background:linear-gradient(135deg,#1E3A5F,#2D5A8E);'
    'border-bottom:4px solid #B91C1C;padding:48px 56px 36px}'
    '.hd .tag{font-size:11px;letter-spacing:2.5px;text-transform:uppercase;'
    'color:#93C5FD;margin-bottom:10px}'
    '.hd h1{font-size:28px;color:#fff;margin-bottom:8px}'
    '.hd p{color:#BFDBFE;font-size:14px}'
    '.hm{display:flex;gap:28px;flex-wrap:wrap;font-size:12px;color:#93C5FD;margin-top:14px}'
    '.tc{display:flex;flex-wrap:wrap;gap:8px;padding:24px 56px;'
    'border-bottom:1px solid var(--bd);background:#172033}'
    '.tc a{color:var(--mt);text-decoration:none;font-size:13px;padding:5px 14px;'
    'border:1px solid var(--bd);border-radius:20px}'
    '.tc a:hover{border-color:#3B82F6;color:#93C5FD}'
    '.pf{display:flex;flex-wrap:wrap;align-items:center;gap:6px;padding:20px 56px;'
    'background:#172033;border-bottom:1px solid var(--bd)}'
    '.fs{background:var(--card);border:1px solid var(--bd);color:var(--tx);'
    'font-size:12px;padding:6px 12px;border-radius:6px;text-decoration:none}'
    '.fs:hover{border-color:#3B82F6;color:#93C5FD}'
    '.fa{color:#475569;font-size:16px}'
    '.ct{max-width:1200px;margin:0 auto;padding:40px 48px 80px}'
    '.sc{margin:48px 0;background:var(--card);border-radius:12px;overflow:hidden;'
    'box-shadow:0 4px 24px rgba(0,0,0,.4)}'
    '.sh{display:flex;align-items:center;gap:18px;padding:20px 28px;background:#172033}'
    '.sn{color:#fff;font-size:11px;font-weight:bold;letter-spacing:1px;'
    'padding:4px 12px;border-radius:4px;white-space:nowrap}'
    '.sh h2{font-size:20px;color:#fff}'
    '.sd{padding:16px 28px;color:var(--mt);font-size:14px;border-bottom:1px solid var(--bd)}'
    '.si{padding:24px 28px;display:flex;flex-direction:column;gap:24px}'
    '.sf{display:flex;flex-direction:column;gap:8px}'
    '.sf img{width:100%;border-radius:8px;border:1px solid var(--bd)}'
    'figcaption{font-size:12px;color:var(--cp);font-style:italic;padding-left:4px}'
    '.miss{color:#EF4444;font-size:13px;padding:16px}'
    'footer{text-align:center;color:var(--cp);font-size:12px;'
    'padding:32px;border-top:1px solid var(--bd)}'
    '@media(max-width:768px){.hd,.ct,.tc,.pf{padding-left:20px;padding-right:20px}}'
)

import datetime as _dt
ts      = _dt.datetime.now().strftime('%Y-%m-%d %H:%M')
mode_lbl = {'synthetic':'Path B — RGB (Synthetic)',
             'real_rgb' :'Path B — Real RGB Camera',
             'real_rgbd':'Path A — RealSense D435 RGB+Depth'}.get(CAPTURE_MODE, CAPTURE_MODE)

html = (
    '<!DOCTYPE html><html lang="en"><head>'
    '<meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1.0">'
    '<title>G18 Pipeline Visual Walkthrough</title>'
    '<style>' + CSS + '</style></head><body>'
    '<div class="hd">'
    '<div class="tag">American University of the Middle East &middot; GP2 Spring 2026</div>'
    '<h1>G18 &mdash; 3D Voronoi Orthopedic Cast Pipeline</h1>'
    '<p>Visual step-by-step walkthrough &mdash; from multi-view photos to print-ready STL files</p>'
    '<div class="hm">'
    '<span>Dr. Georges Zakka El Nashef</span>'
    '<span>Python &middot; OpenCV &middot; Trimesh &middot; manifold3d</span>'
    '<span>' + mode_lbl + '</span>'
    '<span>Generated: ' + ts + '</span>'
    '</div></div>'
    '<nav class="tc">' + toc + '</nav>'
    '<div class="pf">' + flow + '</div>'
    '<div class="ct">' + cards + '</div>'
    '<footer>American University of the Middle East &middot; '
    'College of Engineering &amp; Technology<br>'
    'G18 &mdash; 3D Voronoi Orthopedic Cast System &middot; '
    'GP2 Spring 2026 &middot; Dr. Georges Zakka El Nashef</footer>'
    '</body></html>'
)

html_path = BASE + '/G18_Visual_Walkthrough.html'
with open(html_path,'w',encoding='utf-8') as f:
    f.write(html)
sz = os.path.getsize(html_path)/1e6
print('HTML saved: ' + html_path)
print('Size: {:.1f} MB  (all images embedded as base64)'.format(sz))

from IPython.display import HTML as _H, display
display(_H(
    '<a href="{}" download="G18_Visual_Walkthrough.html" '
    'style="display:inline-block;background:#1E3A5F;color:#93C5FD;'
    'padding:12px 24px;border-radius:8px;text-decoration:none;font-size:14px;'
    'border:1px solid #3B82F6;margin-top:12px;">'
    'Download G18_Visual_Walkthrough.html</a>'.format(html_path)
))